# Causal Inference

In [21]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind

In [5]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

display(hillstrom_df.head(), hillstrom_df.tail())

display(hillstrom_df.info(), hillstrom_df.describe())

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


<class 'pandas.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  str    
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  str    
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  str    
 8   segment          64000 non-null  str    
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), str(4)
memory usage: 5.9 MB


None

,recency,history,mens,womens,newbie,visit,conversion,spend
count,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000
mean,5.763734,242.085656,0.551031,0.549719,0.502250,0.146781,0.009031,1.050908
std,3.507592,256.158608,0.497393,0.497526,0.499999,0.353890,0.094604,15.036448
min,1.000000,29.990000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,64.660000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6.000000,158.110000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,9.000000,325.657500,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
max,12.000000,3345.930000,1.000000,1.000000,1.000000,1.000000,1.000000,499.000000


## Experimental ATE (also calculated in notebook 02)

In [10]:
mens_control_df = hillstrom_df[hillstrom_df["segment"].isin(['Mens E-Mail', 'No E-Mail'])]

In [19]:
# z test for proportions

results = []

for metric in ["visit", "conversion"]:

    ate = mens_control_df[mens_control_df["segment"] == "Mens E-Mail"][metric].mean() - mens_control_df[mens_control_df["segment"] == "No E-Mail"][metric].mean()

    counts_df = mens_control_df.groupby("segment")[metric].agg(["sum", "count"])

    zstat, pvalue = proportions_ztest([counts_df.loc["Mens E-Mail", "sum"], counts_df.loc["No E-Mail", "sum"]],
                                       [counts_df.loc["Mens E-Mail", "count"], counts_df.loc["No E-Mail", "count"]])

    p1 = counts_df.loc["Mens E-Mail", "sum"] / counts_df.loc["Mens E-Mail", "count"]
    p2 = counts_df.loc["No E-Mail", "sum"] / counts_df.loc["No E-Mail", "count"]
    n1 = counts_df.loc["Mens E-Mail", "count"]
    n2 = counts_df.loc["No E-Mail", "count"]
    var_dat_1 = p1 * (1 - p1)
    var_dat_2 = p2 * (1 - p2)
    se_diff = np.sqrt(var_dat_1 / n1 + var_dat_2 / n2)
    ci = (p1 - p2) + np.array([-1, 1]) * 1.96 * se_diff
    ci = ci.round(4)

    results.append({
        "Metric": metric, "ATE": ate, "95% CI": ci, "Z-Statistic": zstat, "P-Value": pvalue, 
    })

pd.DataFrame(results)

,Metric,ATE,95% CI,Z-Statistic,P-Value
0,visit,0.076590,"[0.07, 0.0832]",22.486041,5.685165e-112
1,conversion,0.006805,"[0.005, 0.0086]",7.385114,1.523224e-13


In [23]:
# Spend

spend1 = mens_control_df[mens_control_df["segment"] == "Mens E-Mail"]["spend"]
spend2 = mens_control_df[mens_control_df["segment"] == "No E-Mail"]["spend"]

tstat, pvalue = ttest_ind(spend1, spend2, equal_var=False)

ate = spend1.mean() - spend2.mean()
var_dat_1 = np.sum((spend1 - spend1.mean()) ** 2) / (len(spend1) - 1)
var_dat_2 = np.sum((spend2 - spend2.mean()) ** 2) / (len(spend2) - 1)
var_mean_1 = var_dat_1 / len(spend1)
var_mean_2 = var_dat_2 / len(spend2)
se_diff = np.sqrt(var_mean_1 + var_mean_2)
ci = ate + np.array([-1, 1]) * 1.96 * se_diff
ci = ci.round(4)

results = [{
        "Metric": metric, "ATE": ate, "95% CI": ci, "T-Statistic": tstat, "P-Value": pvalue, 
    }]

pd.DataFrame(results)

,Metric,ATE,95% CI,T-Statistic,P-Value
0,conversion,0.769827,"[0.4851, 1.0545]",5.30014,1.163815e-07


## Introduce artificial selection bias